### Track 3: Agentic Fine-Tuning (AgentTune + AuditKIT)
Ticket triage workflow distillation and GRPO tool-call training.

In [1]:
!pip install -q "auditkit @ git+https://github.com/Lexsi-Labs/AuditKIT.git"
!pip install -q "git+https://github.com/Lexsi-Labs/AgentTune.git" peft accelerate bitsandbytes datasets trl

import os
if not os.path.exists('lexsi-agentic-triage'):
    !git clone https://github.com/prime3436/lexsi-agentic-triage.git
%cd lexsi-agentic-triage

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.

In [2]:
from huggingface_hub import login
tok = os.environ.get('HF_TOKEN')
if tok:
    login(token=tok)

In [3]:
!python generate_dataset.py
!python eval_baseline.py

{'train': 343, 'val': 73, 'test': 74} total 490
{
  "n_test": 74,
  "category_accuracy": 89.2,
  "severity_accuracy": 74.3,
  "specialty_accuracy": 87.8,
  "sla_accuracy": 74.3,
  "hazard_recall": 73.3,
  "hazard_false_positives": 0,
  "avg_latency_ms": 1.69
}


In [4]:
import json
from agenttune.agentic import Project, DictToolHarness, ReActStrategy
from agent import TriageBaseline

ag = TriageBaseline()

def assess_hazard(description: str) -> dict:
    r = ag.run(description)
    return {'hazard_detected': r['hazard_detected'], 'hazard_type': r['hazard_type']}

def classify_trade(description: str) -> dict:
    r = ag.run(description)
    return {'category': r['category'], 'severity': r['severity']}

def estimate_cost(category: str, severity: str) -> dict:
    b = {'P1_CRITICAL': (3500, 9500), 'P2_HIGH': (2800, 7500), 'P3_MEDIUM': (800, 3000), 'P4_LOW': (600, 2000)}
    lo, hi = b.get(severity, (800, 3000))
    return {'min_cost_inr': lo, 'max_cost_inr': hi}

def dispatch_contractor(category: str, severity: str) -> dict:
    s = {'P1_CRITICAL': 2, 'P2_HIGH': 6, 'P3_MEDIUM': 24, 'P4_LOW': 72}
    m = {
        'Plumbing': 'Licensed Master Plumber',
        'Electrical': 'Certified Master Electrician',
        'HVAC / Climate Control': 'EPA-Certified HVAC Technician',
        'Safety & Security': 'Licensed General Handyman',
        'General Maintenance': 'Licensed General Handyman',
    }
    return {'specialty': m.get(category, 'Licensed General Handyman'), 'sla_hours': s.get(severity, 24)}

h = DictToolHarness({
    'assess_hazard': assess_hazard,
    'classify_trade': classify_trade,
    'estimate_cost': estimate_cost,
    'dispatch_contractor': dispatch_contractor,
}, max_steps=6)

def pol(s):
    q = getattr(s, 'task', 'sparking outlet in kitchen')
    if s.step == 0:
        return {'name': 'assess_hazard', 'arguments': {'description': q}, 'thought': 'hazard'}
    if s.step == 1:
        return {'name': 'classify_trade', 'arguments': {'description': q}, 'thought': 'trade'}
    if s.step == 2:
        return {'name': 'estimate_cost', 'arguments': {'category': 'Electrical', 'severity': 'P1_CRITICAL'}, 'thought': 'cost'}
    if s.step == 3:
        return {'name': 'dispatch_contractor', 'arguments': {'category': 'Electrical', 'severity': 'P1_CRITICAL'}, 'thought': 'dispatch'}
    return {'name': 'finish', 'arguments': {'answer': 'P1_CRITICAL Electrical'}}

p = Project(strategy=ReActStrategy(pol, max_steps=6), harness=h)
log = p.infer('The socket in the kitchen is sparking and smells like burning plastic')
print(log.tier, len(log.events))
print(p.evaluate_agentic(['The socket in the kitchen is sparking']))
print(len(p.heal()))

light 20
{'n': 1, 'metrics': {'tac': 0.0, 'ter': 1.0, 'arr': 0.0, 'scsr': 1.0, 'rad': 0.0, 'lcf': 0.0}, 'per_trajectory': [{'tac': 0.0, 'ter': 1.0, 'arr': 0.0, 'scsr': 1.0, 'rad': 0.0, 'lcf': 0}]}
0


In [5]:
from agenttune import create_distill_trainer

rows = [json.loads(l) for l in open('data/agenttune_train.jsonl', encoding='utf-8')]
demos = [(r['prompt'][-1]['content'], r['answer']) for r in rows]
mdl = 'Qwen/Qwen2.5-1.5B-Instruct'

dt = create_distill_trainer(
    student=mdl,
    demonstrations=demos,
    output_dir='./output/distilled',
    num_train_epochs=3,
    learning_rate=2e-4,
)
print(dt.train())

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

... (4 outputs left out)

Building labels for train dataset:   0%|          | 0/343 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 151643}.


Step,Training Loss
10,2.887739
20,1.518563
30,0.852148
40,0.579076
50,0.444987
60,0.407301
70,0.373287
80,0.334624
90,0.297693
100,0.303073


TrainOutput(global_step=129, training_loss=0.6791368417961653, metrics={'train_runtime': 592.9015, 'train_samples_per_second': 1.736, 'train_steps_per_second': 0.218, 'total_flos': 989707849903104.0, 'train_loss': 0.6791368417961653, 'entropy': 0.2940871947341495, 'num_tokens': 117213.0, 'mean_token_accuracy': 0.9397410419252183, 'epoch': 3.0})


In [6]:
from datasets import Dataset
from agenttune.core.backend_factory import create_agentic_trainer

def rew(completions, answer=None, **kw):
    out = []
    for i, c in enumerate(completions):
        txt = str(c)
        g = json.loads(answer[i]) if answer and i < len(answer) else {}
        v = 0.0
        if g.get('category', '') in txt:
            v += 0.35
        if g.get('severity', '') in txt:
            v += 0.35
        if str(g.get('hazard_detected', '')).lower() in txt.lower():
            v += 0.30
        out.append(v)
    return out

ds = Dataset.from_list(rows[:120])

gt = create_agentic_trainer(
    algorithm='grpo',
    model=mdl,
    train_dataset=ds,
    tools=[assess_hazard, classify_trade, estimate_cost, dispatch_contractor],
    reward_funcs=[rew],
    output_dir='./output/grpo',
    max_steps=40,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=2,
    num_generations=2,
    max_steps_per_turn=5,
    use_vllm=False,
    peft_config=dict(r=16, lora_alpha=32, target_modules='all-linear', task_type='CAUSAL_LM'),
    report_to='none',
)
print(gt.train())

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 151643}.
[transformers] Passing `generation_config` together with generation-related arguments=({'disable_compile', 'temperature', 'pad_token_id', 'do_sample', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


Step,Training Loss
10,-0.022967
20,-0.080323
30,-0.039910


Step,Training Loss
10,-0.022967
20,-0.080323
30,-0.039910
40,-0.084623


{'model': 'Qwen/Qwen2.5-1.5B-Instruct', 'dataset_name': 'unknown', 'output_dir': './output/grpo', 'agentic_mode': True, 'tools': ['assess_hazard', 'classify_trade', 'estimate_cost', 'dispatch_contractor'], 'train_size': 120, 'eval_size': 0, 'final_loss': -0.056955694779753686, 'total_steps': 40, 'training_history': [], 'metrics': {'train_runtime': 2268.7919, 'train_samples_per_second': 0.071, 'train_steps_per_second': 0.018, 'total_flos': 0.0, 'train_loss': -0.056955694779753686, 'epoch': 0.6666666666666666}, 'config_kwargs': {'model': 'Qwen/Qwen2.5-1.5B-Instruct', 'train_dataset': "Dataset({\n    features: ['id', 'prompt', 'answer'],\n    num_rows: 120\n})", 'tools': [<function assess_hazard at 0x7fbac69c2a20>, <function classify_trade at 0x7fbac69c2700>, <function estimate_cost at 0x7fb929d54ea0>, <function dispatch_contractor at 0x7fb929daf920>], 'reward_funcs': [<function rew at 0x7fb8a9f714e0>], 'output_dir': './output/grpo', 'max_steps': 40, 'per_device_train_batch_size': 2, 'gra

In [7]:
!python run_auditkit_eval.py

{
  "auditkit_version": "1.1.3",
  "baseline_agent_eval": {
    "system": "baseline",
    "n_cases": 74,
    "verified_outcome_accuracy_pct": 29.7,
    "diagnostics_pct": {
      "tool_call_f1": 85.8,
      "tool_call_exact": 71.6,
      "tool_call_f1_name": 100.0,
      "trajectory_strict": 71.6,
      "trajectory_in_order": 71.6,
      "tool_call_validity": 100.0,
      "redundant_tool_calls": 0.0,
      "agent_loop_detection": 100.0
    }
  },
  "agenttune_agent_eval": {
    "system": "agenttune",
    "n_cases": 74,
    "verified_outcome_accuracy_pct": 100.0,
    "diagnostics_pct": {
      "tool_call_f1": 100.0,
      "tool_call_exact": 100.0,
      "tool_call_f1_name": 100.0,
      "trajectory_strict": 100.0,
      "trajectory_in_order": 100.0,
      "tool_call_validity": 100.0,
      "redundant_tool_calls": 0.0,
      "agent_loop_detection": 100.0
    }
  },
  "auditkit_spine_baseline": {
    "headline": {
      "exact_match": 0.2972972972972973,
      "f1_score": 0.84031792487674

In [8]:
!git pull
!python eval_trained.py
!python eval_trained.py ./output/distilled
!python eval_trained.py ./output/grpo

remote: Enumerating objects: 18, done.
remote: Counting objects: 100% (18/18), done.
remote: Compressing objects: 100% (4/4), done.
remote: Total 10 (delta 6), reused 10 (delta 6), pack-reused 0 (from 0)
Unpacking objects: 100% (10/10), 2.66 KiB | 908.00 KiB/s, done.
From https://github.com/prime3436/lexsi-agentic-triage
   780f197..0635862  main       -> origin/main
Updating 780f197..0635862
Fast-forward
 README.md            | 37 +++++++++++++++----------
 agent.py             |  1 +
 decide_triage.yaml   |  1 +
 eval_baseline.py     |  1 +
 eval_trained.py      | 77 ++++++++++++++++++++++++++++++++++++++++++++++++++++
 generate_dataset.py  |  1 +
 requirements.txt     |  1 +
 run_auditkit_eval.py |  1 +
 8 files changed, 106 insertions(+), 14 deletions(-)
 create mode 100644 eval_trained.py
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 338/338 [00:00<00:00, 401.28it/s]
Some parameters are on the meta device because they were offloaded to the 

In [9]:
%cd /content/lexsi-agentic-triage
!git pull
!ls -R output | head -40
!python eval_trained.py
!python eval_trained.py ./output/distilled
!python eval_trained.py ./output/grpo


/content/lexsi-agentic-triage
remote: Enumerating objects: 5, done.
remote: Counting objects: 100% (5/5), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 3 (delta 2), reused 3 (delta 2), pack-reused 0 (from 0)
Unpacking objects: 100% (3/3), 794 bytes | 794.00 KiB/s, done.
From https://github.com/prime3436/lexsi-agentic-triage
   0635862..952672b  main       -> origin/main
Updating 0635862..952672b
Fast-forward
 eval_trained.py | 34 +++++++++++++++++++++++-----------
 1 file changed, 23 insertions(+), 11 deletions(-)
output:
distilled
grpo

output/distilled:
checkpoint-129
README.md

output/distilled/checkpoint-129:
adapter_config.json
adapter_model.safetensors
chat_template.jinja
optimizer.pt
README.md
rng_state.pth
scheduler.pt
tokenizer_config.json
tokenizer.json
trainer_state.json
training_args.bin

output/grpo:
adapter_config.json
adapter_model.safetensors
chat_template.jinja
checkpoint-40
grpo_training_config.yaml
lexsi_provenance.json
README.md
tokenizer_config

In [10]:

import gc, torch
for k in ["dt", "gt"]:
    if k in globals():
        del globals()[k]
gc.collect()
torch.cuda.empty_cache()

%cd /content/lexsi-agentic-triage
!git pull
!python eval_trained.py
!python eval_trained.py ./output/distilled
!python eval_trained.py ./output/grpo

/content/lexsi-agentic-triage
remote: Enumerating objects: 5, done.
remote: Counting objects: 100% (5/5), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 3 (delta 2), reused 3 (delta 2), pack-reused 0 (from 0)
Unpacking objects: 100% (3/3), 844 bytes | 844.00 KiB/s, done.
From https://github.com/prime3436/lexsi-agentic-triage
   952672b..0be36f6  main       -> origin/main
Updating 952672b..0be36f6
Fast-forward
 eval_trained.py | 41 +++++++++++++++++++++++++++--------------
 1 file changed, 27 insertions(+), 14 deletions(-)
Loading weights: 100% 338/338 [00:00<00:00, 512.97it/s]
Traceback (most recent call last):
  File "/content/lexsi-agentic-triage/eval_trained.py", line 102, in <module>
    main()
    ~~~~^^
  File "/content/lexsi-agentic-triage/eval_trained.py", line 93, in main
    "scores": score(preds, rows),
              ~~~~~^^^^^^^^^^^^^
  File "/content/lexsi-agentic-triage/eval_trained.py", line 75, in score
    tp = sum(p is not None and p.get("hazard_de

In [11]:
!git pull
!python eval_trained.py
!python eval_trained.py ./output/distilled
!python eval_trained.py ./output/grpo

remote: Enumerating objects: 5, done.
remote: Counting objects: 100% (5/5), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 3 (delta 2), reused 3 (delta 2), pack-reused 0 (from 0)
Unpacking objects: 100% (3/3), 411 bytes | 411.00 KiB/s, done.
From https://github.com/prime3436/lexsi-agentic-triage
   0be36f6..aef3876  main       -> origin/main
Updating 0be36f6..aef3876
Fast-forward
 eval_trained.py | 9 +++++----
 1 file changed, 5 insertions(+), 4 deletions(-)
Loading weights: 100% 338/338 [00:00<00:00, 446.87it/s]
{
  "model": "Qwen/Qwen2.5-1.5B-Instruct",
  "scores": {
    "n": 74,
    "valid_json_pct": 29.7,
    "field_accuracy_pct": {
      "category": 0.0,
      "severity": 0.0,
      "hazard_detected": 0.0,
      "sla_hours": 0.0,
      "specialty": 0.0
    },
    "hazard_recall_pct": 0.0
  },
  "auditkit": {
    "headline": {
      "exact_match": 0.0,
      "f1_score": 0.023751418113712564
    },
    "fingerprint": "9cc7660e0ce9ccfb"
  },
  "sample_outputs": [


In [12]:
!git pull
!python eval_trained.py
!python eval_trained.py ./output/distilled
!python eval_trained.py ./output/grpo

remote: Enumerating objects: 5, done.
remote: Counting objects: 100% (5/5), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 3 (delta 2), reused 3 (delta 2), pack-reused 0 (from 0)
Unpacking objects: 100% (3/3), 1.40 KiB | 1.40 MiB/s, done.
From https://github.com/prime3436/lexsi-agentic-triage
   aef3876..3f1c18c  main       -> origin/main
Updating aef3876..3f1c18c
Fast-forward
 eval_trained.py | 104 ++++++++++++++++++++++++++++++++++++++++++++++----------
 1 file changed, 86 insertions(+), 18 deletions(-)
Loading weights: 100% 338/338 [00:00<00:00, 500.22it/s]
{
  "model": "Qwen/Qwen2.5-1.5B-Instruct",
  "scores": {
    "n": 74,
    "valid_json_pct": 29.7,
    "exact_field_accuracy_pct": {
      "category": 0.0,
      "severity": 0.0,
      "hazard_detected": 0.0,
      "sla_hours": 0.0,
      "specialty": 0.0
    },
    "normalized_field_accuracy_pct": {
      "category": 0.0,
      "severity": 0.0,
      "hazard_detected": 0.0,
      "sla_hours": 0.0,
      "speci